In [1]:
import pandas as pd
import torch
import json
import random
import numpy as np

def process_mind_with_memories(behaviors_path, output_path):
    # 1. Load Behavior Data
    cols = ['imp_id', 'user_id', 'time', 'history', 'impressions']
    print("Loading behaviors...")
    df_behav = pd.read_csv(behaviors_path, sep='\t', names=cols)
    # Ensure time is converted to datetime objects for math operations
    df_behav['time'] = pd.to_datetime(df_behav['time'], format='%m/%d/%Y %I:%M:%S %p')

    # 2. Extract Clicks (User Timelines) with 5-minute increments
    def extract_clicks(row):
        parts = row['impressions'].split()
        # Get list of clicked items and negative items
        clicked_ids = [p.split('-')[0] for p in parts if p.endswith('-1')]
        negatives = [p.split('-')[0] for p in parts if p.endswith('-0')]
        
        events = []
        for i, click_id in enumerate(clicked_ids):
            # Calculate offset: 0 min for first click, 5 for second, etc.
            offset = pd.Timedelta(minutes=5 * i)
            adjusted_time = row['time'] + offset
            events.append((adjusted_time, click_id, negatives))
            
        return events

    print("Building user timelines with synthetic 5-min offsets...")
    user_events = {}
    for _, row in df_behav.iterrows():
        events = extract_clicks(row)
        uid = row['user_id']
        if uid not in user_events:
            user_events[uid] = []
        user_events[uid].extend(events)

    # 3. Sliding Window with Slow/Fast Memory
    final_output = []
    FAST_SIZE = 20
    MIN_SLOW = 10
    MIN_TOTAL = MIN_SLOW + FAST_SIZE + 1 # 31 clicks

    print(f"Generating sequences (Min total clicks: {MIN_TOTAL})...")

    for uid, events in user_events.items():
        # Sort chronologically to ensure the offsets and different impressions align
        events.sort(key=lambda x: x[0])
        
        if len(events) < MIN_TOTAL:
            continue

        for i in range(MIN_TOTAL - 1, len(events)):
            target_event = events[i]
            target_timestamp = str(target_event[0])
            target_id = target_event[1]
            neg_ids = target_event[2]
            
            # Fast Memory: The 20 items exactly before target
            fast_window = events[i - FAST_SIZE : i]
            fast_ids = [e[1] for e in fast_window]
            fast_id_timestamps = [str(e[0]) for e in fast_window]
            
            # Slow Memory: Everything before the fast window
            slow_window = events[0 : i - FAST_SIZE]
            slow_ids = [e[1] for e in slow_window]
            slow_id_timestamps = [str(e[0]) for e in slow_window]

            if len(neg_ids) > 100:
                neg_ids = random.sample(neg_ids, 100)
            
            # Construct entry
            entry = {
                "user_id": uid,
                "slow_memory_ids": slow_ids,
                "slow_id_timestamps": slow_id_timestamps,
                "fast_memory_ids": fast_ids,
                "fast_id_timestamps": fast_id_timestamps,
                "target_item": target_id,
                "target_timestamp": target_timestamp,
                "negative_items": neg_ids
            }
            
            final_output.append(entry)

    # 4. Save to JSON
    print(f"Saving {len(final_output)} sequences...")
    with open(output_path, 'w') as f:
        json.dump(final_output, f)
    
    print("Done!")

# Execution
process_mind_with_memories(
    '/kaggle/input/datasets/xuntngtrn/mind-large-train/behaviors.tsv', 
    'processed_mind_memories.json'
)

Loading behaviors...
Building user timelines with synthetic 5-min offsets...
Generating sequences (Min total clicks: 31)...
Saving 71616 sequences...
Done!


# CATW - Balanced Version (Fine-tuning + Controlled Regularization)

**Key design:**
- ✓ Fine-tuned embeddings at low LR (task-specific adaptation)
- ✓ 1 transformer block (conservative capacity to prevent overfitting)
- ✓ Dropout 0.35 (strong but not excessive)
- ✓ Pure BPR loss (no conflicting objectives)
- ✓ Early stop on validation loss plateau (patience=5)
- ✓ Load best checkpoint for testing


In [2]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import json, pickle
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## Data Loading

In [3]:
# Load news data
df = pd.read_csv('/kaggle/input/datasets/xuntngtrn/mind-large-train/news.tsv', sep='\t', header=None)
df.columns = ['News_ID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'Title_Entities', 'Abstract_Entities']
df = df[~df['Category'].isin(['games', 'middleeast', 'northamerica'])]
df['Abstract'] = df['Abstract'].fillna('')
df['Full_Text'] = df['Title'] + " . " + df['Abstract']

news_id_to_int = {sid: idx for idx, sid in enumerate(df['News_ID'].unique(), start=1)}
category_to_int = {cat: idx for idx, cat in enumerate(df['Category'].unique(), start=1)}
df['Category_Int'] = df['Category'].map(category_to_int)
news_string_to_cat_int = dict(zip(df['News_ID'], df['Category_Int']))

print(f"Items: {len(news_id_to_int)}, Categories: {len(category_to_int)}")

# Load embeddings
try:
    pretrained_matrix = torch.load('mind_pretrained_matrix.pt')
except:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    d_model = encoder.get_sentence_embedding_dimension()
    pretrained_matrix = torch.zeros((len(news_id_to_int) + 1, d_model))
    embeddings = encoder.encode(df['Full_Text'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True).cpu()
    for idx, sid in enumerate(df['News_ID'].tolist()):
        pretrained_matrix[news_id_to_int[sid]] = embeddings[idx]
    torch.save(pretrained_matrix, 'mind_pretrained_matrix.pt')

# Load behavioral data
with open('processed_mind_memories.json', 'r') as f:
    data = json.load(f)

mind_dataset = []
for d in data:
    all_ids = d['slow_memory_ids'] + d['fast_memory_ids'] + d['negative_items'] + [d['target_item']]
    if all(item_id in news_id_to_int for item_id in all_ids):
        mind_dataset.append(d)

train_data, temp = train_test_split(mind_dataset, test_size=5000, random_state=42)
val_data, test_data = train_test_split(temp, test_size=0.5, random_state=42)
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Items: 101523, Categories: 15


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1587 [00:00<?, ?it/s]

Train: 66544, Val: 2500, Test: 2500


In [4]:
class RecDataset(Dataset):
    def __init__(self, data_list, id2int, cat_id2int):
        self.data = data_list
        self.id2int = id2int
        self.cat_id2int = cat_id2int

    def __len__(self):
        return len(self.data)

    def _parse_time(self, s):
        return datetime.strptime(s, '%Y-%m-%d %H:%M:%S')

    def __getitem__(self, idx):
        row = self.data[idx]
        hist_ids = row['slow_memory_ids'] + row['fast_memory_ids']
        hist_times = row['slow_id_timestamps'] + row['fast_id_timestamps']

        history = [self.id2int.get(i, 0) for i in hist_ids]
        hist_cats = [self.cat_id2int.get(i, 0) for i in hist_ids]

        t_q = self._parse_time(row['target_timestamp'])
        hist_dt = [max(0, (t_q - self._parse_time(t)).total_seconds() / 3600) for t in hist_times]

        pos = self.id2int.get(row['target_item'], 0)
        negs = [self.id2int.get(n, 0) for n in row['negative_items'][:99]]

        return {
            'hist': torch.tensor(history, dtype=torch.long),
            'hist_cat': torch.tensor(hist_cats, dtype=torch.long),
            'hist_dt': torch.tensor(hist_dt, dtype=torch.float),
            'pos': torch.tensor(pos, dtype=torch.long),
            'neg': torch.tensor(negs, dtype=torch.long)
        }

def collate(batch):
    return {
        'hist': pad_sequence([b['hist'] for b in batch], batch_first=True),
        'hist_cat': pad_sequence([b['hist_cat'] for b in batch], batch_first=True),
        'hist_dt': pad_sequence([b['hist_dt'] for b in batch], batch_first=True),
        'pos': torch.stack([b['pos'] for b in batch]),
        'neg': pad_sequence([b['neg'] for b in batch], batch_first=True)
    }

train_ds = RecDataset(train_data, news_id_to_int, news_string_to_cat_int)
val_ds = RecDataset(val_data, news_id_to_int, news_string_to_cat_int)
test_ds = RecDataset(test_data, news_id_to_int, news_string_to_cat_int)

BS = 32  # Standard batch size
train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True, collate_fn=collate, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BS, shuffle=False, collate_fn=collate, num_workers=0)
test_dl = DataLoader(test_ds, batch_size=BS, shuffle=False, collate_fn=collate, num_workers=0)

print(f"DataLoaders ready (BS={BS})")

DataLoaders ready (BS=32)


## Model: 1 Block + Fine-tuning

In [5]:
class TemporalAttention(nn.Module):
    def __init__(self, d, h, drop=0.35):
        super().__init__()
        self.d, self.h, self.dk = d, h, d // h
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        self.gamma = nn.Parameter(torch.tensor(1.0))
        for w in [self.q, self.k, self.v, self.o]:
            nn.init.xavier_uniform_(w.weight)
            nn.init.zeros_(w.bias)

    def forward(self, x, dt_p, mask=None):
        B, T = x.size(0), x.size(1)
        Q = self.q(x).view(B, T, self.h, self.dk).transpose(1, 2)
        K = self.k(x).view(B, T, self.h, self.dk).transpose(1, 2)
        V = self.v(x).view(B, T, self.h, self.dk).transpose(1, 2)
        sc = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.dk)
        gam = F.softplus(self.gamma)
        pen = gam * torch.log1p(dt_p).unsqueeze(1).unsqueeze(2)
        sc = sc - pen
        if mask is not None:
            sc = sc.masked_fill(mask == 0, -1e9)
        att = F.softmax(sc, dim=-1)
        att = torch.nan_to_num(att, 0)
        att = self.drop(att)
        ctx = torch.matmul(att, V).transpose(1, 2).contiguous().view(B, T, self.d)
        return self.o(ctx)


class CATW(nn.Module):
    def __init__(self, ni, nc, d, h, emb=None, k=20, drop=0.35, nb=1):
        super().__init__()
        self.d, self.k = d, k

        # Item embeddings - FINE-TUNABLE at lower LR
        self.emb = nn.Embedding(ni, d, padding_idx=0)
        if emb is not None:
            self.emb.weight.data.copy_(emb)
        self.emb.weight.requires_grad = True  # Allow fine-tuning

        # Category embeddings
        self.cat_emb = nn.Embedding(nc, 16, padding_idx=0)
        nn.init.xavier_uniform_(self.cat_emb.weight[1:])

        # Decay network
        self.decay_mlp = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(8, 1)
        )

        # Position embeddings
        self.pos_emb = nn.Embedding(k + 1, d)
        nn.init.xavier_uniform_(self.pos_emb.weight)

        # Transformer (1 block)
        self.attn = nn.ModuleList([TemporalAttention(d, h, drop) for _ in range(nb)])
        self.ln1 = nn.ModuleList([nn.LayerNorm(d) for _ in range(nb)])
        self.ln2 = nn.ModuleList([nn.LayerNorm(d) for _ in range(nb)])
        self.ffn = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d*4), nn.GELU(), nn.Dropout(drop),
                nn.Linear(d*4, d)
            ) for _ in range(nb)
        ])
        self.drop = nn.Dropout(drop)

    def forward(self, h, hc, hdt, p=None, n=None):
        B, T = h.size()

        # Get embeddings
        x = self.emb(h)  # [B, T, D]

        # Decay rates
        ce = self.cat_emb(hc)
        dec = F.softplus(self.decay_mlp(ce)).squeeze(-1)
        dt_p = dec * hdt

        # Padding mask
        mp = (h != 0).float()
        dtp_m = dt_p.clone()
        dtp_m[mp == 0] = 1e6

        # Dynamic eviction
        k = min(self.k, T)
        dtf, idxf = torch.topk(dtp_m, k, dim=1, largest=False)

        # Fast memory
        fast_x = torch.gather(x, 1, idxf.unsqueeze(-1).expand(-1, -1, self.d))

        # Persona from slow memory
        fm = torch.zeros_like(h, dtype=torch.bool).scatter_(1, idxf, True)
        sm = (mp > 0) & (~fm)
        slow_x = x * sm.unsqueeze(-1).float()
        slow_cnt = sm.sum(1, keepdim=True).float().clamp(min=1)
        pers = slow_x.sum(1) / slow_cnt

        # Combine
        seq = torch.cat([pers.unsqueeze(1), fast_x], dim=1)  # [B, k+1, D]
        sq = k + 1

        dtp = torch.zeros(B, 1, device=h.device)
        dta = torch.cat([dtp, dtf], dim=1)

        # Pos emb + dropout
        pos_idx = torch.arange(sq, device=h.device).unsqueeze(0).expand(B, -1)
        seq = seq + self.pos_emb(pos_idx)
        seq = self.drop(seq)

        # Causal mask
        mc = torch.tril(torch.ones(sq, sq, device=h.device))

        # Transformer
        for a, l1, l2, f in zip(self.attn, self.ln1, self.ln2, self.ffn):
            ax = a(l1(seq), dta, mc)
            seq = seq + self.drop(ax)
            fx = f(l2(seq))
            seq = seq + self.drop(fx)

        rep = seq[:, -1, :]

        if p is not None:
            pe = self.emb(p)
            ps = (rep * pe).sum(1)
            if n is not None:
                ne = self.emb(n)
                ns = torch.bmm(ne, rep.unsqueeze(2)).squeeze(2)
                return ps, ns
            return ps
        return None


D = pretrained_matrix.size(1)
H = 4
NI = pretrained_matrix.size(0)
NC = len(category_to_int) + 1

model = CATW(NI, NC, D, H, pretrained_matrix, k=20, drop=0.35, nb=1).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} total params")
print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")

Model: 40,768,146 total params
Trainable: 40,768,146 params


## Training: BPR Loss + Early Stopping on Validation Loss

In [6]:
import numpy as np
from sklearn.metrics import roc_auc_score

def get_ranks(ps, ns):
    """Helper to get 0-indexed ranks of positive items"""
    # all_s shape: [B, 1 + N]
    all_s = torch.cat([ps.unsqueeze(1), ns], dim=1)
    # Sort descending: higher scores get lower rank indices
    # We find where index 0 ended up after sorting
    ranked = torch.argsort(all_s, dim=1, descending=True)
    pos_rank = (ranked == 0).nonzero(as_tuple=True)[1]
    return pos_rank + 1, all_s # returns 1-based rank and scores

def compute_metrics(ps, ns, ks=[5, 10]):
    ranks, all_s = get_ranks(ps, ns)
    ranks_float = ranks.float()
    
    # 1. Mean MRR
    mrr = (1.0 / ranks_float).mean().item()
    
    # 2. NDCG@K
    metrics = {f'ndcg@{k}': 0 for k in ks}
    for k in ks:
        # NDCG = 1/log2(rank + 1) if rank <= k else 0
        ndcg_mask = (ranks <= k).float()
        ndcg = (ndcg_mask / torch.log2(ranks_float + 1)).mean().item()
        metrics[f'ndcg@{k}'] = ndcg
    
    # 3. Group AUC
    # GAUC is essentially the average AUC per user
    # Since we have 1 pos and N negs per group:
    # AUC = (number of negs with score < pos score) / total negs
    # Equivalent to: ( (1 + N) - rank ) / N
    num_neg = ns.size(1)
    gauc = ((num_neg + 1 - ranks_float) / num_neg).mean().item()
    
    metrics['mrr'] = mrr
    metrics['gauc'] = gauc
    return metrics

In [7]:
def bpr_loss(ps, ns):
    """BPR: maximize margin between pos and neg"""
    diff = ps.unsqueeze(1) - ns  # [B, N]
    return -torch.log(torch.sigmoid(diff) + 1e-8).mean()


def hr_at_k(ps, ns, k=10):
    """Proper HR@k: Is positive in top-k among all candidates?"""
    all_s = torch.cat([ps.unsqueeze(1), ns], dim=1)
    ranked = torch.argsort(all_s, dim=1, descending=True)
    pos_rank = (ranked == 0).nonzero(as_tuple=True)[1]
    return (pos_rank < k).float().mean().item()


# Differential learning rates: slower for embeddings, faster for task layers
opt = AdamW([
    {'params': model.emb.parameters(), 'lr': 3e-4},  # Conservative for embeddings
    {'params': [p for name, p in model.named_parameters() if 'emb' not in name], 'lr': 2e-3}
], weight_decay=2e-4)  # Moderate L2

sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.7, patience=3, min_lr=1e-6
)

best_vl = float('inf')
best_ep = 0
eat_cnt = 0

for ep in range(1, 80):
    # Train
    model.train()
    tl, thr = 0, 0
    nb = 0

    for bat in tqdm(train_dl, desc=f"E{ep} [Train]", leave=False):
        h, hc, hdt = bat['hist'].to(device), bat['hist_cat'].to(device), bat['hist_dt'].to(device)
        p, n = bat['pos'].to(device), bat['neg'].to(device)

        opt.zero_grad()
        ps, ns = model(h, hc, hdt, p, n)
        loss = bpr_loss(ps, ns)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tl += loss.item()
        thr += hr_at_k(ps, ns, k=10)
        nb += 1

    atl = tl / nb
    athr = thr / nb

    # Val
    model.eval()
    vl, vhr = 0, 0
    nb = 0

    with torch.no_grad():
        for bat in val_dl:
            h, hc, hdt = bat['hist'].to(device), bat['hist_cat'].to(device), bat['hist_dt'].to(device)
            p, n = bat['pos'].to(device), bat['neg'].to(device)
            ps, ns = model(h, hc, hdt, p, n)
            loss = bpr_loss(ps, ns)
            vl += loss.item()
            vhr += hr_at_k(ps, ns, k=10)
            nb += 1

    avl = vl / nb
    avhr = vhr / nb

    # Early stop on validation LOSS (not HR), patience=5
    st = " "
    if avl < best_vl:
        best_vl = avl
        best_ep = ep
        torch.save(model.state_dict(), "best_catw_balanced.pth")
        st = "✓"
        eat_cnt = 0
    else:
        eat_cnt += 1

    sch.step(avl)
    
    gap = athr - avhr
    print(f"E{ep:2d} | Tr: L={atl:.4f} HR={athr:.4f} | Val: L={avl:.4f} HR={avhr:.4f} | Gap={gap:.4f} {st}")

    # Stop when validation loss increases for 5 consecutive epochs
    if eat_cnt >= 5:
        print(f"\nEarly stop @ E{ep} (best: E{best_ep} VL={best_vl:.4f})")
        break

print(f"\nTraining complete. Best validation loss: {best_vl:.4f}")

E 1 | Tr: L=0.6929 HR=0.4032 | Val: L=0.4007 HR=0.4810 | Gap=-0.0778 ✓


E 2 | Tr: L=0.3746 HR=0.5004 | Val: L=0.3730 HR=0.5222 | Gap=-0.0217 ✓


E 3 | Tr: L=0.3372 HR=0.5266 | Val: L=0.3773 HR=0.5146 | Gap=0.0120  


E 4 | Tr: L=0.3161 HR=0.5456 | Val: L=0.3822 HR=0.5301 | Gap=0.0156  


E 5 | Tr: L=0.2992 HR=0.5604 | Val: L=0.3919 HR=0.5245 | Gap=0.0358  


E 6 | Tr: L=0.2836 HR=0.5799 | Val: L=0.4082 HR=0.5336 | Gap=0.0463  


E 7 | Tr: L=0.2557 HR=0.6132 | Val: L=0.4074 HR=0.5253 | Gap=0.0879  

Early stop @ E7 (best: E2 VL=0.3730)

Training complete. Best validation loss: 0.3730


## Test with Best Checkpoint

In [8]:
model.load_state_dict(torch.load("best_catw_balanced.pth"))
model.eval()

# Initialize accumulators for all metrics
test_metrics = {
    'loss': 0, 'gauc': 0, 'mrr': 0, 
    'hr@10': 0, 'ndcg@5': 0, 'ndcg@10': 0
}
nb = 0

with torch.no_grad():
    for bat in tqdm(test_dl, desc="Testing"):
        h, hc, hdt = bat['hist'].to(device), bat['hist_cat'].to(device), bat['hist_dt'].to(device)
        p, n = bat['pos'].to(device), bat['neg'].to(device)
        
        ps, ns = model(h, hc, hdt, p, n)
        
        # Calculate loss
        loss = bpr_loss(ps, ns)
        
        # Calculate all metrics using the function from previous step
        m = compute_metrics(ps, ns, ks=[5, 10])
        
        # Accumulate
        test_metrics['loss'] += loss.item()
        test_metrics['gauc'] += m['gauc']
        test_metrics['mrr'] += m['mrr']
        test_metrics['hr@10'] += hr_at_k(ps, ns, k=10) # Keeping your original HR function
        test_metrics['ndcg@5'] += m['ndcg@5']
        test_metrics['ndcg@10'] += m['ndcg@10']
        nb += 1

# Average the results
for k in test_metrics:
    test_metrics[k] /= nb

print(f"\n{'='*50}")
print(f"🎯 TEST RESULTS (from best validation checkpoint)")
print(f"{'='*50}")
print(f"Loss:      {test_metrics['loss']:.4f}")
print(f"Group AUC: {test_metrics['gauc']:.4f}")
print(f"Mean MRR:  {test_metrics['mrr']:.4f}")
print(f"HR@10:     {test_metrics['hr@10']:.4f}")
print(f"NDCG@5:    {test_metrics['ndcg@5']:.4f}")
print(f"NDCG@10:   {test_metrics['ndcg@10']:.4f}")

# Performance evaluation based on HR@10
thr = test_metrics['hr@10']

Testing: 100%|██████████| 79/79 [00:01<00:00, 62.00it/s]


🎯 TEST RESULTS (from best validation checkpoint)
Loss:      0.3706
Group AUC: 0.8299
Mean MRR:  0.2436
HR@10:     0.5138
NDCG@5:    0.2375
NDCG@10:   0.2901


## Analysis: Learned Decay Rates

In [9]:
with torch.no_grad():
    decays = {}
    for cid, cname in enumerate(category_to_int.keys(), 1):
        ct = torch.tensor([cid], device=device)
        ce = model.cat_emb(ct)
        dec = F.softplus(model.decay_mlp(ce)).item()
        decays[cname] = dec

print("\n📊 Learned Decay Rates (λ_c):")
print("="*60)
for c, d in sorted(decays.items(), key=lambda x: x[1]):
    print(f"{c:20s}: {d:6.4f}")
print("="*60)
print(f"\nInterpretation:")
print(f"  Low λ_c  → Interest decays slowly (durable categories)")
print(f"  High λ_c → Interest decays quickly (timely categories)")


📊 Learned Decay Rates (λ_c):
sports              : 0.0190
video               : 0.0512
health              : 0.0605
news                : 0.0654
lifestyle           : 0.0941
autos               : 0.0979
tv                  : 0.0990
music               : 0.1000
entertainment       : 0.1042
movies              : 0.1311
travel              : 0.1914
finance             : 0.1963
weather             : 0.2271
kids                : 0.2634
foodanddrink        : 0.2994

Interpretation:
  Low λ_c  → Interest decays slowly (durable categories)
  High λ_c → Interest decays quickly (timely categories)
